# 1. Purpose, scope, and phase mapping

This notebook tests whether the residual-boundary finding is wholly dependent on the broad `Others` class. It removes all `Others`-involving pairs, re-ranks the six remaining named-class boundaries across the original ten seeds and the existing 2,000 bootstrap replicates per seed, and audits the already-frozen hard-pair set.

Under the working assumption that this dataset was not re-audited or relabelled, annotation sensitivity is not applicable. This is therefore operational Phase 6, corresponding to the original master plan's Phase 7 `Others` sensitivity analysis.

The notebook runs on CPU. It does not load models, read raw text or prediction ZIPs, use a network, access the test set, or change the ordered Phase 5 hard-pair set `H`.


# 2. User configuration


In [1]:
# USER CONFIGURATION
from pathlib import Path

PHASE4_RESULTS_DIR = Path("phase4_boundary_analysis_results")
PHASE5_RESULTS_DIR = Path("phase5_bootstrap_stability_results")
OUTPUT_DIR = Path("phase6_others_sensitivity_results")
RESULT_ZIP_PATH = Path("MentalDistress_Phase6_Others_Sensitivity_Results.zip")

PHASE4_PAIR_METRICS_PATH = PHASE4_RESULTS_DIR / "phase4" / "phase4_per_seed_pair_metrics.csv"
PHASE5_BOOTSTRAP_PATH = PHASE5_RESULTS_DIR / "phase5_per_replicate_pair_metrics.csv"
PHASE5_SELECTION_PATH = PHASE5_RESULTS_DIR / "phase5_hard_pair_selection.json"
PHASE5_PROTOCOL_PATH = PHASE5_RESULTS_DIR / "phase5_protocol.json"


# 3. Environment


In [2]:
import os
import platform
import json
import sys
import tempfile
import time

import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

os.environ["PYTHONHASHSEED"] = "0"
np.random.seed(0)
environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "nbformat": nbformat.__version__,
    "execution_device": "CPU",
    "network_required": False,
}
print(json.dumps(environment, indent=2, sort_keys=True))


{
  "execution_device": "CPU",
  "nbformat": "5.10.4",
  "network_required": false,
  "numpy": "2.5.1",
  "pandas": "3.0.3",
  "platform": "Linux-7.0.0-30-generic-x86_64-with-glibc2.43",
  "python": "3.14.4"
}


# 4. Embedded implementation

The full implementation is embedded so the notebook has no project-local Python-module dependency.


In [3]:
"""Phase 6 sensitivity analysis excluding the broad `Others` class."""

from __future__ import annotations

import hashlib
import html
import json
from itertools import combinations
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd


NAMED_CLASSES = ["Suicidal", "Depressed", "Anxious", "Frustrated"]
NAMED_PAIRS = [f"{left}|{right}" for left, right in combinations(NAMED_CLASSES, 2)]
EXPECTED_SEEDS = 10
EXPECTED_REPLICATES = 2000
EXPECTED_ORIGINAL_ROWS = 60
EXPECTED_BOOTSTRAP_ROWS = 120000
DIAGNOSTIC_TOP3_COUNT = 7
DIAGNOSTIC_BOOTSTRAP_FREQUENCY = 0.70


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def validate_manifest(root):
    """Verify every file declared by an existing result manifest."""
    root = Path(root)
    manifest_path = root / "sha256_manifest.csv"
    if not manifest_path.is_file():
        raise ValueError(f"manifest is absent: {manifest_path}")
    manifest = pd.read_csv(manifest_path)
    if list(manifest.columns) != ["path", "sha256", "bytes"] or manifest.empty:
        raise ValueError("manifest schema is invalid or empty")
    for row in manifest.itertuples(index=False):
        path = root / str(row.path)
        if not path.is_file():
            raise ValueError(f"manifest file is absent: {row.path}")
        if path.stat().st_size != int(row.bytes):
            raise ValueError(f"manifest byte-count mismatch: {row.path}")
        if sha256_file(path) != str(row.sha256):
            raise ValueError(f"manifest SHA-256 mismatch: {row.path}")
    return {"status": "PASS", "root": str(root), "manifest_entries": int(len(manifest))}


def _validate_named_group_sizes(frame, group_columns):
    sizes = frame.groupby(group_columns, sort=False).size()
    if sizes.empty or not (sizes == len(NAMED_PAIRS)).all():
        raise ValueError("every analysis group must contain exactly six named pairs")
    if frame.groupby(group_columns, sort=False)["pair"].nunique().ne(len(NAMED_PAIRS)).any():
        raise ValueError("every analysis group must contain six named pairs uniquely")


def filter_and_rerank_original(pair_metrics):
    """Filter Phase 4 metrics and recompute six-pair ranks per seed."""
    required = {"seed", "pair", "bcr"}
    missing = required.difference(pair_metrics.columns)
    if missing:
        raise ValueError(f"Phase 4 table missing columns: {sorted(missing)}")
    frame = pair_metrics.loc[pair_metrics["pair"].isin(NAMED_PAIRS)].copy()
    _validate_named_group_sizes(frame, ["seed"])
    if not np.isfinite(frame["bcr"].to_numpy(dtype=float)).all():
        raise ValueError("original BCR values must be finite")
    frame["named_rank"] = frame.groupby("seed", sort=False)["bcr"].rank(
        method="average", ascending=False
    )
    frame["is_top1_named"] = frame["named_rank"] <= 1.0
    frame["is_top3_named"] = frame["named_rank"] <= 3.0
    return frame.sort_values(["seed", "named_rank", "pair"], kind="mergesort").reset_index(drop=True)


def filter_and_rerank_bootstrap(bootstrap_metrics):
    """Filter Phase 5 observations and recompute ranks per seed/replicate."""
    required = {"model_seed", "bootstrap_replicate", "pair", "bcr"}
    missing = required.difference(bootstrap_metrics.columns)
    if missing:
        raise ValueError(f"Phase 5 table missing columns: {sorted(missing)}")
    frame = bootstrap_metrics.loc[bootstrap_metrics["pair"].isin(NAMED_PAIRS)].copy()
    groups = ["model_seed", "bootstrap_replicate"]
    _validate_named_group_sizes(frame, groups)
    if not np.isfinite(frame["bcr"].to_numpy(dtype=float)).all():
        raise ValueError("bootstrap BCR values must be finite")
    frame["named_rank"] = frame.groupby(groups, sort=False)["bcr"].rank(
        method="average", ascending=False
    )
    frame["is_top1_named"] = frame["named_rank"] <= 1.0
    frame["is_top3_named"] = frame["named_rank"] <= 3.0
    return frame.sort_values(groups + ["named_rank", "pair"], kind="mergesort").reset_index(drop=True)


def summarize_original(restricted):
    rows = []
    for pair, group in restricted.groupby("pair", sort=True):
        bcr = group["bcr"].to_numpy(dtype=float)
        ranks = group["named_rank"].to_numpy(dtype=float)
        rows.append(
            {
                "pair": pair,
                "seed_count": int(len(group)),
                "mean_bcr": float(np.mean(bcr)),
                "sample_sd_bcr": float(np.std(bcr, ddof=1)) if len(bcr) > 1 else 0.0,
                "median_bcr": float(np.median(bcr)),
                "bcr_q1": float(np.quantile(bcr, 0.25, method="linear")),
                "bcr_q3": float(np.quantile(bcr, 0.75, method="linear")),
                "mean_rank": float(np.mean(ranks)),
                "median_rank": float(np.median(ranks)),
                "top1_count": int(np.sum(ranks <= 1.0)),
                "top1_frequency": float(np.mean(ranks <= 1.0)),
                "top3_count": int(np.sum(ranks <= 3.0)),
                "top3_frequency": float(np.mean(ranks <= 3.0)),
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["median_rank", "mean_bcr", "pair"], ascending=[True, False, True], kind="mergesort"
    ).reset_index(drop=True)


def _kendalls_w(rank_matrix):
    values = rank_matrix.to_numpy(dtype=float)
    pair_count, seed_count = values.shape
    expected_sum = pair_count * (pair_count + 1) / 2
    if pair_count < 2 or seed_count < 2 or not np.isfinite(values).all():
        raise ValueError("rank matrix must contain finite ranks for at least two pairs and seeds")
    if not np.allclose(values.sum(axis=0), expected_sum, rtol=0.0, atol=1e-9):
        raise ValueError("rank columns are incomplete")
    rank_sums = values.sum(axis=1)
    centered = float(np.sum((rank_sums - seed_count * (pair_count + 1) / 2) ** 2))
    tie_correction = 0.0
    for column in values.T:
        _, counts = np.unique(column, return_counts=True)
        tie_correction += float(np.sum(counts**3 - counts))
    denominator = seed_count**2 * (pair_count**3 - pair_count) - seed_count * tie_correction
    if denominator <= 0:
        raise ValueError("Kendall W denominator is not positive")
    return {
        "method": "Kendall coefficient of concordance with tie correction",
        "kendalls_w": float(np.clip(12 * centered / denominator, 0, 1)),
        "pair_count": int(pair_count),
        "seed_count": int(seed_count),
        "tie_correction": float(tie_correction),
    }


def original_rank_agreement(restricted):
    rank_matrix = restricted.pivot(index="pair", columns="seed", values="named_rank")
    rank_matrix = rank_matrix.reindex(sorted(rank_matrix.index)).reindex(columns=sorted(rank_matrix.columns))
    rows = []
    for seed_a, seed_b in combinations(rank_matrix.columns.tolist(), 2):
        rho = float(np.corrcoef(rank_matrix[seed_a], rank_matrix[seed_b])[0, 1])
        if not np.isfinite(rho):
            raise ValueError("pairwise Spearman value is not finite")
        rows.append({"seed_a": int(seed_a), "seed_b": int(seed_b), "spearman_rho": rho})
    comparisons = pd.DataFrame(rows)
    agreement = _kendalls_w(rank_matrix)
    rho = comparisons["spearman_rho"].to_numpy(dtype=float)
    agreement["pairwise_spearman"] = {
        "method": "Pearson correlation applied to complete average-rank vectors",
        "comparison_count": int(len(comparisons)),
        "median_spearman_rho": float(np.median(rho)),
        "spearman_rho_q1": float(np.quantile(rho, 0.25, method="linear")),
        "spearman_rho_q3": float(np.quantile(rho, 0.75, method="linear")),
        "minimum_spearman_rho": float(np.min(rho)),
        "maximum_spearman_rho": float(np.max(rho)),
    }
    return rank_matrix, agreement, comparisons


def summarize_bootstrap_seed_pairs(restricted):
    rows = []
    for (seed, pair), group in restricted.groupby(["model_seed", "pair"], sort=True):
        bcr = group["bcr"].to_numpy(dtype=float)
        ranks = group["named_rank"].to_numpy(dtype=float)
        rows.append(
            {
                "model_seed": int(seed),
                "pair": pair,
                "bootstrap_replicates": int(len(group)),
                "bcr_mean": float(np.mean(bcr)),
                "bcr_sample_sd": float(np.std(bcr, ddof=1)) if len(bcr) > 1 else 0.0,
                "bcr_median": float(np.median(bcr)),
                "bcr_q1": float(np.quantile(bcr, 0.25, method="linear")),
                "bcr_q3": float(np.quantile(bcr, 0.75, method="linear")),
                "bcr_ci_lower": float(np.quantile(bcr, 0.025, method="linear")),
                "bcr_ci_upper": float(np.quantile(bcr, 0.975, method="linear")),
                "top1_count": int(np.sum(ranks <= 1.0)),
                "top1_frequency": float(np.mean(ranks <= 1.0)),
                "top3_count": int(np.sum(ranks <= 3.0)),
                "top3_frequency": float(np.mean(ranks <= 3.0)),
                "median_rank": float(np.median(ranks)),
            }
        )
    return pd.DataFrame(rows).sort_values(["model_seed", "pair"], kind="mergesort").reset_index(drop=True)


def summarize_bootstrap_pairs(seed_summary, restricted):
    rows = []
    for pair, group in seed_summary.groupby("pair", sort=True):
        pooled = restricted.loc[restricted["pair"] == pair, "bcr"].to_numpy(dtype=float)
        rows.append(
            {
                "pair": pair,
                "seed_count": int(len(group)),
                "mean_seed_bcr_mean": float(group["bcr_mean"].mean()),
                "mean_seed_top1_frequency": float(group["top1_frequency"].mean()),
                "mean_seed_top3_frequency": float(group["top3_frequency"].mean()),
                "median_seed_median_rank": float(group["median_rank"].median()),
                "pooled_observation_count": int(len(pooled)),
                "pooled_bcr_mean": float(np.mean(pooled)),
                "pooled_bcr_median": float(np.median(pooled)),
                "pooled_bcr_q1": float(np.quantile(pooled, 0.25, method="linear")),
                "pooled_bcr_q3": float(np.quantile(pooled, 0.75, method="linear")),
                "pooled_bcr_ci_lower": float(np.quantile(pooled, 0.025, method="linear")),
                "pooled_bcr_ci_upper": float(np.quantile(pooled, 0.975, method="linear")),
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["mean_seed_top3_frequency", "mean_seed_bcr_mean", "pair"],
        ascending=[False, False, True], kind="mergesort"
    ).reset_index(drop=True)


def bootstrap_rank_distribution(restricted):
    counts = (
        restricted.groupby(["model_seed", "pair", "named_rank"], sort=True)
        .size().rename("count").reset_index()
    )
    totals = counts.groupby(["model_seed", "pair"])["count"].transform("sum")
    counts["frequency"] = counts["count"] / totals
    return counts


def error_concentration(pair_metrics):
    required = {"seed", "pair", "pair_errors", "pair_error_share", "total_errors"}
    missing = required.difference(pair_metrics.columns)
    if missing:
        raise ValueError(f"pair table missing error columns: {sorted(missing)}")
    rows = []
    for seed, group in pair_metrics.groupby("seed", sort=True):
        named = group["pair"].isin(NAMED_PAIRS)
        named_errors = int(group.loc[named, "pair_errors"].sum())
        others_errors = int(group.loc[~named, "pair_errors"].sum())
        total_errors = int(group["total_errors"].iloc[0])
        if group["total_errors"].nunique() != 1 or named_errors + others_errors != total_errors:
            raise ValueError("pair errors do not partition total errors")
        named_share = float(group.loc[named, "pair_error_share"].sum())
        others_share = float(group.loc[~named, "pair_error_share"].sum())
        if total_errors and not np.isclose(named_share + others_share, 1.0, rtol=0.0, atol=1e-12):
            raise ValueError("named and Others pair-error shares do not sum to one")
        named_group = group.loc[named].sort_values(
            ["pair_errors", "bcr", "pair"], ascending=[False, False, True], kind="mergesort"
        )
        leader = named_group.iloc[0]
        rows.append(
            {
                "seed": int(seed),
                "total_errors": total_errors,
                "named_pair_errors": named_errors,
                "others_pair_errors": others_errors,
                "named_pair_error_share": named_share,
                "others_pair_error_share": others_share,
                "leading_named_pair": leader["pair"],
                "leading_named_pair_errors": int(leader["pair_errors"]),
                "leading_named_share_of_named_errors": (
                    float(leader["pair_errors"] / named_errors) if named_errors else 0.0
                ),
            }
        )
    per_seed = pd.DataFrame(rows)
    measures = [
        "total_errors", "named_pair_errors", "others_pair_errors",
        "named_pair_error_share", "others_pair_error_share",
        "leading_named_pair_errors", "leading_named_share_of_named_errors",
    ]
    summary_rows = []
    for measure in measures:
        values = per_seed[measure].to_numpy(dtype=float)
        summary_rows.append(
            {
                "measure": measure,
                "seed_count": int(len(values)),
                "mean": float(np.mean(values)),
                "sample_sd": float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
                "median": float(np.median(values)),
                "minimum": float(np.min(values)),
                "maximum": float(np.max(values)),
            }
        )
    return per_seed, pd.DataFrame(summary_rows)


def audit_frozen_h(frozen_h, original_summary, bootstrap_summary):
    frozen = list(frozen_h)
    others = [pair for pair in frozen if "Others" in pair.split("|")]
    named = [pair for pair in frozen if pair in NAMED_PAIRS]
    original_lookup = original_summary.set_index("pair")
    bootstrap_lookup = bootstrap_summary.set_index("pair")
    details = []
    for pair in named:
        if pair not in original_lookup.index or pair not in bootstrap_lookup.index:
            raise ValueError(f"frozen named pair is absent from restricted summaries: {pair}")
        original_top3 = int(original_lookup.loc[pair, "top3_count"])
        bootstrap_top3 = float(bootstrap_lookup.loc[pair, "mean_seed_top3_frequency"])
        details.append(
            {
                "pair": pair,
                "restricted_original_top3_count": original_top3,
                "restricted_bootstrap_mean_seed_top3_frequency": bootstrap_top3,
                "diagnostic_original_pass": original_top3 >= DIAGNOSTIC_TOP3_COUNT,
                "diagnostic_bootstrap_pass": bootstrap_top3 >= DIAGNOSTIC_BOOTSTRAP_FREQUENCY,
                "remains_stable_descriptively": (
                    original_top3 >= DIAGNOSTIC_TOP3_COUNT
                    and bootstrap_top3 >= DIAGNOSTIC_BOOTSTRAP_FREQUENCY
                ),
            }
        )
    return {
        "H_before": frozen,
        "H_after": list(frozen),
        "H_unchanged": True,
        "H_count": int(len(frozen)),
        "others_involving_pairs": others,
        "others_involving_count": int(len(others)),
        "others_involving_fraction": float(len(others) / len(frozen)) if frozen else 0.0,
        "non_others_pairs": named,
        "non_others_count": int(len(named)),
        "non_others_pair_diagnostics": details,
        "diagnostic_thresholds": {
            "restricted_original_top3_minimum_seed_count": DIAGNOSTIC_TOP3_COUNT,
            "restricted_bootstrap_minimum_mean_seed_top3_frequency": DIAGNOSTIC_BOOTSTRAP_FREQUENCY,
        },
        "interpretation": "Sensitivity diagnostics do not select, remove, reorder, or replace any member of H.",
        "test_accessed": False,
    }


def run_phase6(phase4_pair_metrics, phase5_bootstrap_metrics, phase5_selection):
    original = filter_and_rerank_original(phase4_pair_metrics)
    original_summary = summarize_original(original)
    rank_matrix, agreement, spearman = original_rank_agreement(original)
    bootstrap = filter_and_rerank_bootstrap(phase5_bootstrap_metrics)
    seed_summary = summarize_bootstrap_seed_pairs(bootstrap)
    pooled_summary = summarize_bootstrap_pairs(seed_summary, bootstrap)
    rank_distribution = bootstrap_rank_distribution(bootstrap)
    error_per_seed, error_summary = error_concentration(phase4_pair_metrics)
    h_audit = audit_frozen_h(phase5_selection["H"], original_summary, pooled_summary)
    protocol = {
        "phase": "Phase 6 — Others Sensitivity",
        "original_master_plan_mapping": "Phase 7 — Others Sensitivity",
        "renumbering_reason": "Annotation sensitivity is not applicable under the working assumption that the dataset was not re-audited or relabelled.",
        "named_classes": NAMED_CLASSES,
        "named_pairs": NAMED_PAIRS,
        "original_seed_count": int(original["seed"].nunique()),
        "bootstrap_replicates_per_seed": int(bootstrap["bootstrap_replicate"].nunique()),
        "rank_method": "descending BCR with average ranks for exact ties",
        "top1_rule": "named_rank <= 1",
        "top3_rule": "named_rank <= 3",
        "bcr_policy": "Preserve original BCR values and full true-class support denominators; change only the ranking universe.",
        "frozen_h_policy": "Read Phase 5 H and preserve its ordered contents unchanged.",
        "execution_device": "CPU",
        "test_accessed": False,
    }
    return {
        "protocol": protocol,
        "original_named_pair_metrics": original,
        "original_named_pair_summary": original_summary,
        "named_seed_rank_matrix": rank_matrix,
        "named_rank_agreement": agreement,
        "named_pairwise_seed_spearman": spearman,
        "bootstrap_named_pair_metrics": bootstrap,
        "bootstrap_seed_pair_summary": seed_summary,
        "bootstrap_pair_summary": pooled_summary,
        "bootstrap_rank_distribution": rank_distribution,
        "error_concentration_per_seed": error_per_seed,
        "error_concentration_summary": error_summary,
        "frozen_h_sensitivity": h_audit,
    }


def validate_phase6_results(results, expected_seeds=EXPECTED_SEEDS,
                            expected_replicates=EXPECTED_REPLICATES):
    original = results["original_named_pair_metrics"]
    bootstrap = results["bootstrap_named_pair_metrics"]
    if len(original) != int(expected_seeds) * 6:
        raise ValueError("restricted original row count mismatch")
    if len(bootstrap) != int(expected_seeds) * int(expected_replicates) * 6:
        raise ValueError("restricted bootstrap row count mismatch")
    if original["seed"].nunique() != int(expected_seeds):
        raise ValueError("original seed count mismatch")
    replicate_counts = bootstrap.groupby("model_seed")["bootstrap_replicate"].nunique()
    if len(replicate_counts) != int(expected_seeds) or not (replicate_counts == int(expected_replicates)).all():
        raise ValueError("bootstrap replicate count mismatch")
    if not (bootstrap.groupby("pair").size() == int(expected_seeds) * int(expected_replicates)).all():
        raise ValueError("bootstrap per-pair count mismatch")
    if len(results["bootstrap_seed_pair_summary"]) != int(expected_seeds) * 6:
        raise ValueError("bootstrap seed-pair summary count mismatch")
    if len(results["bootstrap_pair_summary"]) != 6:
        raise ValueError("bootstrap pair summary count mismatch")
    if len(results["named_pairwise_seed_spearman"]) != 45:
        raise ValueError("pairwise seed comparison count mismatch")
    for frame in [original, bootstrap]:
        if not np.isfinite(frame[["bcr", "named_rank"]].to_numpy(dtype=float)).all():
            raise ValueError("restricted BCR/rank values must be finite")
    if not np.array_equal(bootstrap["is_top1_named"], bootstrap["named_rank"] <= 1):
        raise ValueError("restricted bootstrap top-1 flags disagree with ranks")
    if not np.array_equal(bootstrap["is_top3_named"], bootstrap["named_rank"] <= 3):
        raise ValueError("restricted bootstrap top-3 flags disagree with ranks")
    for frame in [results["bootstrap_seed_pair_summary"], results["bootstrap_pair_summary"]]:
        for column in [name for name in frame.columns if name.endswith("frequency")]:
            values = frame[column].to_numpy(dtype=float)
            if ((values < 0) | (values > 1)).any():
                raise ValueError(f"frequency outside [0, 1]: {column}")
    audit = results["frozen_h_sensitivity"]
    if not audit["H_unchanged"] or audit["H_before"] != audit["H_after"]:
        raise ValueError("frozen H changed during sensitivity analysis")
    return {
        "status": "PASS",
        "original_named_pair_rows": int(len(original)),
        "bootstrap_named_pair_rows": int(len(bootstrap)),
        "seed_count": int(expected_seeds),
        "bootstrap_replicates_per_seed": int(expected_replicates),
        "named_pair_count": 6,
        "bootstrap_observations_per_pair": int(expected_seeds) * int(expected_replicates),
        "bootstrap_seed_pair_summary_rows": int(len(results["bootstrap_seed_pair_summary"])),
        "bootstrap_pair_summary_rows": int(len(results["bootstrap_pair_summary"])),
        "pairwise_spearman_rows": int(len(results["named_pairwise_seed_spearman"])),
        "finite_metrics": True,
        "rank_flags_consistent": True,
        "frozen_h_unchanged": True,
        "test_accessed": False,
    }


def _json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"not JSON serializable: {type(value).__name__}")


def _write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=_json_default) + "\n", encoding="utf-8"
    )


def _write_dataframe(path, frame):
    frame.to_csv(path, index=False, float_format="%.17g", lineterminator="\n")


def _svg_document(title, body, width=1000, height=700):
    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">\n'
        '<rect width="100%" height="100%" fill="white"/>\n'
        '<style>text{font-family:Arial,sans-serif;fill:#172033}.title{font-size:24px;font-weight:700}'
        '.label{font-size:14px}.small{font-size:12px}.axis{stroke:#65728a;stroke-width:1}</style>\n'
        f'<text x="{width/2:.0f}" y="34" text-anchor="middle" class="title">{html.escape(title)}</text>\n'
        f'{body}\n</svg>\n'
    )


def _write_figures(results, figures_dir):
    figures_dir = Path(figures_dir)
    figures_dir.mkdir(parents=True, exist_ok=True)
    original = results["original_named_pair_summary"].sort_values("mean_bcr", ascending=False)
    lines = []
    maximum = max(float(original["mean_bcr"].max()), 1e-12)
    for index, row in enumerate(original.itertuples(index=False)):
        y = 90 + index * 80
        width = 600 * row.mean_bcr / maximum
        lines += [
            f'<text x="290" y="{y+21}" text-anchor="end" class="label">{html.escape(row.pair)}</text>',
            f'<rect x="300" y="{y}" width="{width:.2f}" height="30" fill="#2f6f9f"/>',
            f'<text x="{310+width:.2f}" y="{y+21}" class="label">{row.mean_bcr:.4f}</text>',
        ]
    (figures_dir / "original_named_mean_bcr.svg").write_text(
        _svg_document("Named-class boundary BCR across ten seeds", "\n".join(lines)), encoding="utf-8"
    )

    bootstrap = results["bootstrap_pair_summary"].sort_values(
        "mean_seed_top3_frequency", ascending=False
    )
    lines = []
    for index, row in enumerate(bootstrap.itertuples(index=False)):
        y = 90 + index * 80
        width = 600 * row.mean_seed_top3_frequency
        lines += [
            f'<text x="290" y="{y+21}" text-anchor="end" class="label">{html.escape(row.pair)}</text>',
            f'<rect x="300" y="{y}" width="{width:.2f}" height="30" fill="#327a5f"/>',
            f'<text x="{310+width:.2f}" y="{y+21}" class="label">{row.mean_seed_top3_frequency:.3f}</text>',
        ]
    (figures_dir / "bootstrap_named_top3_frequency.svg").write_text(
        _svg_document("Restricted bootstrap top-3 stability", "\n".join(lines)), encoding="utf-8"
    )

    errors = results["error_concentration_per_seed"].sort_values("seed")
    lines = ['<line x1="170" y1="600" x2="930" y2="600" class="axis"/>']
    bar_width = 58
    for index, row in enumerate(errors.itertuples(index=False)):
        x = 190 + index * 72
        named_height = 450 * row.named_pair_error_share
        others_height = 450 * row.others_pair_error_share
        lines += [
            f'<rect x="{x}" y="{600-named_height:.2f}" width="{bar_width}" height="{named_height:.2f}" fill="#2f6f9f"/>',
            f'<rect x="{x}" y="{600-named_height-others_height:.2f}" width="{bar_width}" height="{others_height:.2f}" fill="#d17a22"/>',
            f'<text x="{x+bar_width/2:.1f}" y="625" text-anchor="middle" class="small">{row.seed}</text>',
        ]
    lines += [
        '<rect x="760" y="70" width="18" height="18" fill="#2f6f9f"/><text x="785" y="84" class="label">Named-pair errors</text>',
        '<rect x="760" y="98" width="18" height="18" fill="#d17a22"/><text x="785" y="112" class="label">Others-involving errors</text>',
    ]
    (figures_dir / "error_concentration_by_seed.svg").write_text(
        _svg_document("Validation-error concentration by pair type", "\n".join(lines)), encoding="utf-8"
    )


def write_sha256_manifest(root):
    root = Path(root)
    output = root / "sha256_manifest.csv"
    rows = []
    for path in sorted(item for item in root.rglob("*") if item.is_file() and item != output):
        rows.append({
            "path": path.relative_to(root).as_posix(),
            "sha256": sha256_file(path),
            "bytes": int(path.stat().st_size),
        })
    pd.DataFrame(rows, columns=["path", "sha256", "bytes"]).to_csv(
        output, index=False, lineterminator="\n"
    )
    return rows


def write_phase6_results(results, output_dir, validation, source_checks, create_figures=True):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    _write_json(output_dir / "phase6_protocol.json", results["protocol"])
    table_map = {
        "phase6_original_named_pair_metrics.csv": "original_named_pair_metrics",
        "phase6_original_named_pair_summary.csv": "original_named_pair_summary",
        "phase6_named_pairwise_seed_spearman.csv": "named_pairwise_seed_spearman",
        "phase6_bootstrap_named_pair_metrics.csv": "bootstrap_named_pair_metrics",
        "phase6_bootstrap_seed_pair_summary.csv": "bootstrap_seed_pair_summary",
        "phase6_bootstrap_pair_summary.csv": "bootstrap_pair_summary",
        "phase6_bootstrap_rank_distribution.csv": "bootstrap_rank_distribution",
        "phase6_error_concentration_per_seed.csv": "error_concentration_per_seed",
        "phase6_error_concentration_summary.csv": "error_concentration_summary",
    }
    for filename, key in table_map.items():
        _write_dataframe(output_dir / filename, results[key])
    _write_dataframe(
        output_dir / "phase6_named_seed_rank_matrix.csv",
        results["named_seed_rank_matrix"].reset_index(),
    )
    _write_json(output_dir / "phase6_named_rank_agreement.json", results["named_rank_agreement"])
    _write_json(output_dir / "phase6_frozen_h_sensitivity.json", results["frozen_h_sensitivity"])
    if create_figures:
        _write_figures(results, output_dir / "figures")
    original_leader = results["original_named_pair_summary"].iloc[0]
    bootstrap_leader = results["bootstrap_pair_summary"].iloc[0]
    error_lookup = results["error_concentration_summary"].set_index("measure")
    named_h = results["frozen_h_sensitivity"]["non_others_pair_diagnostics"]
    named_h_lines = "\n".join(
        f"- `{item['pair']}`: original top-3 {item['restricted_original_top3_count']}/10; "
        f"bootstrap top-3 {item['restricted_bootstrap_mean_seed_top3_frequency']:.4f}; "
        f"remains stable: {str(item['remains_stable_descriptively']).lower()}"
        for item in named_h
    ) or "- No non-`Others` pair is present in frozen H."
    report = (
        "# Phase 6 `Others` Sensitivity Report\n\n"
        "## Material Passport\n\n"
        "- Sources: manifest-verified frozen Phase 4 and Phase 5 result tables.\n"
        "- Statistical units: model seed for original analysis; model-seed/bootstrap-replicate for resampling analysis.\n"
        "- Scope: six unordered pairs among Suicidal, Depressed, Anxious, and Frustrated.\n"
        "- Phase mapping: operational Phase 6, corresponding to original master-plan Phase 7.\n"
        "- Verification: source manifests, schemas, counts, ranks, frequencies, frozen-H equality, output manifest, and deterministic ZIP.\n"
        "- Test access: false. No models, raw text, prediction ZIPs, GPU, or network were used.\n"
        "- Limitation: this sensitivity analysis changes the ranking universe but preserves BCR denominators; it does not redefine H.\n\n"
        "## Main results\n\n"
        f"- Leading original named boundary: `{original_leader.pair}` (mean BCR {original_leader.mean_bcr:.6f}).\n"
        f"- Leading restricted-bootstrap boundary: `{bootstrap_leader.pair}` "
        f"(mean seed top-3 frequency {bootstrap_leader.mean_seed_top3_frequency:.4f}).\n"
        f"- Kendall's W across original six-pair rankings: {results['named_rank_agreement']['kendalls_w']:.6f}.\n"
        f"- Mean named-pair error share: {error_lookup.loc['named_pair_error_share', 'mean']:.4f}.\n"
        f"- Mean `Others`-involving error share: {error_lookup.loc['others_pair_error_share', 'mean']:.4f}.\n\n"
        "## Frozen-H audit\n\n"
        f"Frozen H contains {results['frozen_h_sensitivity']['others_involving_count']} `Others`-involving pair(s) "
        f"out of {results['frozen_h_sensitivity']['H_count']}.\n\n"
        f"{named_h_lines}\n\n"
        "The ordered Phase 5 hard-pair set was not changed. Composition dependence and residual named-class structure are reported separately.\n"
    )
    (output_dir / "phase6_report.md").write_text(report, encoding="utf-8")
    execution = {
        **validation,
        "source_manifest_checks": source_checks,
        "top_original_named_pair": str(original_leader.pair),
        "top_bootstrap_named_pair": str(bootstrap_leader.pair),
        "kendalls_w": float(results["named_rank_agreement"]["kendalls_w"]),
        "mean_named_pair_error_share": float(error_lookup.loc["named_pair_error_share", "mean"]),
        "mean_others_pair_error_share": float(error_lookup.loc["others_pair_error_share", "mean"]),
        "frozen_h": results["frozen_h_sensitivity"]["H_after"],
        "final_hard_pairs_frozen": True,
        "execution_device": "CPU",
    }
    _write_json(output_dir / "phase6_execution_summary.json", execution)
    manifest = write_sha256_manifest(output_dir)
    return {"execution_summary": execution, "manifest_entries": int(len(manifest))}


def write_deterministic_zip(source_dir, zip_path, prefix="phase6_others_sensitivity_results"):
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)
    fixed_time = (2020, 1, 1, 0, 0, 0)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
        for path in sorted(item for item in source_dir.rglob("*") if item.is_file()):
            member = (Path(prefix) / path.relative_to(source_dir)).as_posix()
            info = zipfile.ZipInfo(member, fixed_time)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o644 << 16
            archive.writestr(info, path.read_bytes())
    return zip_path


# 5. Synthetic self-tests


In [4]:
_all_classes = [*NAMED_CLASSES, "Others"]
_toy_rows = []
for _index, (_left, _right) in enumerate(combinations(_all_classes, 2), start=1):
    _toy_rows.append({
        "seed": 13,
        "pair": f"{_left}|{_right}",
        "class_i": _left,
        "class_j": _right,
        "bcr": (11 - _index) / 100,
        "rank": 99.0,
        "pair_errors": 11 - _index,
        "pair_error_share": (11 - _index) / 55,
        "total_errors": 55,
    })
_toy_original = pd.DataFrame(_toy_rows)
_toy_named = filter_and_rerank_original(_toy_original)
assert len(_toy_named) == 6
assert set(_toy_named["pair"]) == set(NAMED_PAIRS)
assert _toy_named["named_rank"].max() <= 6

_toy_bootstrap = _toy_original.rename(
    columns={"seed": "model_seed", "class_i": "class_a", "class_j": "class_b"}
).copy()
_toy_bootstrap["bootstrap_replicate"] = 1
_toy_bootstrap["rank"] = 99.0
_toy_bootstrap["is_top1"] = False
_toy_bootstrap["is_top3"] = False
_toy_bootstrap_named = filter_and_rerank_bootstrap(_toy_bootstrap)
assert not (_toy_bootstrap_named["named_rank"] == 99).any()
assert np.array_equal(
    _toy_bootstrap_named["is_top3_named"],
    _toy_bootstrap_named["named_rank"] <= 3,
)

# Exact ties receive average ranks.
_tie = _toy_original.copy()
_tie.loc[_tie["pair"] == "Depressed|Anxious", "bcr"] = _tie.loc[
    _tie["pair"] == "Suicidal|Depressed", "bcr"
].iat[0]
_tie_rank = filter_and_rerank_original(_tie).set_index("pair")["named_rank"]
assert _tie_rank["Suicidal|Depressed"] == 1.5
assert _tie_rank["Depressed|Anxious"] == 1.5

_toy_errors, _ = error_concentration(_toy_original)
assert abs(_toy_errors.loc[0, "named_pair_error_share"] + _toy_errors.loc[0, "others_pair_error_share"] - 1) < 1e-12
print("Embedded Phase 6 synthetic self-tests: PASS")


Embedded Phase 6 synthetic self-tests: PASS


# 6. Source-manifest and input gates


In [5]:
source_checks = {
    "phase4": validate_manifest(PHASE4_RESULTS_DIR),
    "phase5": validate_manifest(PHASE5_RESULTS_DIR),
}
phase4_pair_metrics = pd.read_csv(PHASE4_PAIR_METRICS_PATH)
phase5_bootstrap_metrics = pd.read_csv(PHASE5_BOOTSTRAP_PATH)
phase5_selection = json.loads(PHASE5_SELECTION_PATH.read_text(encoding="utf-8"))
phase5_protocol = json.loads(PHASE5_PROTOCOL_PATH.read_text(encoding="utf-8"))

assert len(phase4_pair_metrics) == 100
assert phase4_pair_metrics["seed"].nunique() == 10
assert phase4_pair_metrics["pair"].nunique() == 10
assert len(phase5_bootstrap_metrics) == 200000
assert phase5_bootstrap_metrics["model_seed"].nunique() == 10
assert phase5_bootstrap_metrics["pair"].nunique() == 10
assert phase5_bootstrap_metrics.groupby("model_seed")["bootstrap_replicate"].nunique().eq(2000).all()
assert set(phase4_pair_metrics["seed"]) == set(phase5_bootstrap_metrics["model_seed"])
assert set(phase4_pair_metrics["pair"]) == set(phase5_bootstrap_metrics["pair"])
assert phase5_selection["test_accessed"] is False
assert phase5_protocol["test_accessed"] is False
assert phase5_protocol["bootstrap_replicates_per_seed"] == 2000
print(json.dumps(source_checks, indent=2, sort_keys=True))
print("Phase 4/5 source integrity and schema gates: PASS")


{
  "phase4": {
    "manifest_entries": 17,
    "root": "phase4_boundary_analysis_results",
    "status": "PASS"
  },
  "phase5": {
    "manifest_entries": 12,
    "root": "phase5_bootstrap_stability_results",
    "status": "PASS"
  }
}
Phase 4/5 source integrity and schema gates: PASS


# 7. Six-pair original analysis


In [6]:
analysis_started = time.perf_counter()
phase6 = run_phase6(phase4_pair_metrics, phase5_bootstrap_metrics, phase5_selection)
original_named = phase6["original_named_pair_metrics"]
assert len(original_named) == 60
assert original_named["seed"].nunique() == 10
assert original_named["pair"].nunique() == 6
assert len(phase6["named_pairwise_seed_spearman"]) == 45
display(phase6["original_named_pair_summary"])
print(json.dumps(phase6["named_rank_agreement"], indent=2, sort_keys=True))


,pair,seed_count,mean_bcr,sample_sd_bcr,median_bcr,bcr_q1,bcr_q3,mean_rank,median_rank,top1_count,top1_frequency,top3_count,top3_frequency
0,Suicidal|Depressed,10,0.078162,0.004147,0.077880,0.076534,0.080163,1.0,1.0,10,1.0,10,1.0
1,Depressed|Anxious,10,0.026351,0.004783,0.025312,0.023903,0.029442,2.2,2.0,0,0.0,10,1.0
2,Suicidal|Frustrated,10,0.021361,0.003776,0.020472,0.018026,0.023750,3.0,3.0,0,0.0,8,0.8
3,Depressed|Frustrated,10,0.015763,0.003207,0.014548,0.013711,0.018065,3.9,4.0,0,0.0,2,0.2
4,Suicidal|Anxious,10,0.010341,0.003239,0.009021,0.008237,0.011538,5.0,5.0,0,0.0,0,0.0
5,Anxious|Frustrated,10,0.006321,0.001233,0.006161,0.005311,0.007011,5.9,6.0,0,0.0,0,0.0


{
  "kendalls_w": 0.9348571428571428,
  "method": "Kendall coefficient of concordance with tie correction",
  "pair_count": 6,
  "pairwise_spearman": {
    "comparison_count": 45,
    "maximum_spearman_rho": 1.0,
    "median_spearman_rho": 0.942857142857143,
    "method": "Pearson correlation applied to complete average-rank vectors",
    "minimum_spearman_rho": 0.8285714285714287,
    "spearman_rho_q1": 0.8857142857142858,
    "spearman_rho_q3": 0.942857142857143
  },
  "seed_count": 10,
  "tie_correction": 0.0
}


# 8. Six-pair bootstrap reanalysis


In [7]:
bootstrap_named = phase6["bootstrap_named_pair_metrics"]
assert len(bootstrap_named) == 120000
assert bootstrap_named.groupby("model_seed")["bootstrap_replicate"].nunique().eq(2000).all()
assert bootstrap_named.groupby("pair").size().eq(20000).all()
validation = validate_phase6_results(
    phase6,
    expected_seeds=EXPECTED_SEEDS,
    expected_replicates=EXPECTED_REPLICATES,
)
display(phase6["bootstrap_pair_summary"])


,pair,seed_count,mean_seed_bcr_mean,mean_seed_top1_frequency,mean_seed_top3_frequency,median_seed_median_rank,pooled_observation_count,pooled_bcr_mean,pooled_bcr_median,pooled_bcr_q1,pooled_bcr_q3,pooled_bcr_ci_lower,pooled_bcr_ci_upper
0,Suicidal|Depressed,10,0.078087,0.99940,1.00000,1.0,20000,0.078087,0.077726,0.069309,0.086586,0.053907,0.103556
1,Depressed|Anxious,10,0.026333,0.00045,0.88530,2.0,20000,0.026333,0.025603,0.020455,0.031870,0.011369,0.044451
2,Suicidal|Frustrated,10,0.021357,0.00010,0.70515,3.0,20000,0.021357,0.020948,0.016008,0.025888,0.009025,0.036480
3,Depressed|Frustrated,10,0.015782,0.00005,0.32580,4.0,20000,0.015782,0.015026,0.011308,0.019592,0.005415,0.030051
4,Suicidal|Anxious,10,0.010398,0.00000,0.07825,5.0,20000,0.010398,0.009804,0.006536,0.013206,0.001701,0.021979
5,Anxious|Frustrated,10,0.006345,0.00000,0.00550,6.0,20000,0.006345,0.005415,0.003506,0.008712,0.000000,0.014127


# 9. Error concentration and frozen-H audit


In [8]:
frozen_h_before = list(phase5_selection["H"])
frozen_h_after = list(phase6["frozen_h_sensitivity"]["H_after"])
assert frozen_h_after == frozen_h_before
assert phase6["frozen_h_sensitivity"]["H_unchanged"] is True
assert np.allclose(
    phase6["error_concentration_per_seed"]["named_pair_error_share"]
    + phase6["error_concentration_per_seed"]["others_pair_error_share"],
    1.0,
    rtol=0.0,
    atol=1e-12,
)
display(phase6["error_concentration_per_seed"])
print(json.dumps(phase6["frozen_h_sensitivity"], indent=2, sort_keys=True))


,seed,total_errors,named_pair_errors,others_pair_errors,named_pair_error_share,others_pair_error_share,leading_named_pair,leading_named_pair_errors,leading_named_share_of_named_errors
0,13,190,79,111,0.415789,0.584211,Suicidal|Depressed,39,0.493671
1,42,199,83,116,0.417085,0.582915,Suicidal|Depressed,47,0.566265
2,77,195,84,111,0.430769,0.569231,Suicidal|Depressed,38,0.452381
3,123,205,78,127,0.380488,0.619512,Suicidal|Depressed,40,0.512821
4,211,193,85,108,0.440415,0.559585,Suicidal|Depressed,38,0.447059
5,340,202,80,122,0.396040,0.603960,Suicidal|Depressed,37,0.462500
6,512,199,82,117,0.412060,0.587940,Suicidal|Depressed,37,0.451220
7,1024,198,82,116,0.414141,0.585859,Suicidal|Depressed,36,0.439024
8,2026,203,86,117,0.423645,0.576355,Suicidal|Depressed,40,0.465116
9,3407,204,83,121,0.406863,0.593137,Suicidal|Depressed,38,0.457831


{
  "H_after": [
    "Frustrated|Others",
    "Suicidal|Depressed",
    "Depressed|Others"
  ],
  "H_before": [
    "Frustrated|Others",
    "Suicidal|Depressed",
    "Depressed|Others"
  ],
  "H_count": 3,
  "H_unchanged": true,
  "diagnostic_thresholds": {
    "restricted_bootstrap_minimum_mean_seed_top3_frequency": 0.7,
    "restricted_original_top3_minimum_seed_count": 7
  },
  "interpretation": "Sensitivity diagnostics do not select, remove, reorder, or replace any member of H.",
  "non_others_count": 1,
  "non_others_pair_diagnostics": [
    {
      "diagnostic_bootstrap_pass": true,
      "diagnostic_original_pass": true,
      "pair": "Suicidal|Depressed",
      "remains_stable_descriptively": true,
      "restricted_bootstrap_mean_seed_top3_frequency": 1.0,
      "restricted_original_top3_count": 10
    }
  ],
  "non_others_pairs": [
    "Suicidal|Depressed"
  ],
  "others_involving_count": 2,
  "others_involving_fraction": 0.6666666666666666,
  "others_involving_pairs": [
   

# 10. Results and figures


In [9]:
write_state = write_phase6_results(
    phase6,
    OUTPUT_DIR,
    validation=validation,
    source_checks=source_checks,
    create_figures=True,
)
required_outputs = [
    "phase6_protocol.json",
    "phase6_original_named_pair_metrics.csv",
    "phase6_original_named_pair_summary.csv",
    "phase6_named_seed_rank_matrix.csv",
    "phase6_named_rank_agreement.json",
    "phase6_named_pairwise_seed_spearman.csv",
    "phase6_bootstrap_named_pair_metrics.csv",
    "phase6_bootstrap_seed_pair_summary.csv",
    "phase6_bootstrap_pair_summary.csv",
    "phase6_bootstrap_rank_distribution.csv",
    "phase6_error_concentration_per_seed.csv",
    "phase6_error_concentration_summary.csv",
    "phase6_frozen_h_sensitivity.json",
    "phase6_execution_summary.json",
    "phase6_report.md",
    "sha256_manifest.csv",
]
for name in required_outputs:
    assert (OUTPUT_DIR / name).is_file(), name
print(f"Wrote {write_state['manifest_entries']} manifest-covered Phase 6 files.")


Wrote 18 manifest-covered Phase 6 files.


# 11. Manifest and deterministic ZIP


In [10]:
output_manifest_check = validate_manifest(OUTPUT_DIR)
write_deterministic_zip(OUTPUT_DIR, RESULT_ZIP_PATH)
with tempfile.TemporaryDirectory() as temporary_directory:
    comparison_zip = Path(temporary_directory) / "comparison.zip"
    write_deterministic_zip(OUTPUT_DIR, comparison_zip)
    assert sha256_file(RESULT_ZIP_PATH) == sha256_file(comparison_zip)
print("Output manifest and deterministic ZIP verification: PASS")


Output manifest and deterministic ZIP verification: PASS


# 12. Final status


In [11]:
analysis_elapsed_seconds = time.perf_counter() - analysis_started
final_summary = json.loads((OUTPUT_DIR / "phase6_execution_summary.json").read_text(encoding="utf-8"))
assert final_summary["status"] == "PASS"
assert final_summary["test_accessed"] is False
assert final_summary["frozen_h_unchanged"] is True
assert final_summary["original_named_pair_rows"] == 60
assert final_summary["bootstrap_named_pair_rows"] == 120000
assert final_summary["bootstrap_observations_per_pair"] == 20000

print(json.dumps({
    **final_summary,
    "analysis_elapsed_seconds": round(analysis_elapsed_seconds, 3),
    "result_zip": str(RESULT_ZIP_PATH),
    "result_zip_bytes": RESULT_ZIP_PATH.stat().st_size,
    "result_zip_sha256": sha256_file(RESULT_ZIP_PATH),
}, indent=2, sort_keys=True))
print("PHASE 6 OTHERS SENSITIVITY: PASS")


{
  "analysis_elapsed_seconds": 3.322,
  "bootstrap_named_pair_rows": 120000,
  "bootstrap_observations_per_pair": 20000,
  "bootstrap_pair_summary_rows": 6,
  "bootstrap_replicates_per_seed": 2000,
  "bootstrap_seed_pair_summary_rows": 60,
  "execution_device": "CPU",
  "final_hard_pairs_frozen": true,
  "finite_metrics": true,
  "frozen_h": [
    "Frustrated|Others",
    "Suicidal|Depressed",
    "Depressed|Others"
  ],
  "frozen_h_unchanged": true,
  "kendalls_w": 0.9348571428571428,
  "mean_named_pair_error_share": 0.4137295829143618,
  "mean_others_pair_error_share": 0.5862704170856377,
  "named_pair_count": 6,
  "original_named_pair_rows": 60,
  "pairwise_spearman_rows": 45,
  "rank_flags_consistent": true,
  "result_zip": "MentalDistress_Phase6_Others_Sensitivity_Results.zip",
  "result_zip_bytes": 1191521,
  "result_zip_sha256": "33b11811899950354db6e3173913d4f1633943a193752c543c4a661ea02db10a",
  "seed_count": 10,
  "source_manifest_checks": {
    "phase4": {
      "manifest_e